HSE, Applied Time Series Forecasitng , Winter 2024-25
____


# <font color="green"> Lesson #7: Measuring Forecast Accuracy. Loss Functions in Forecasting</font>
____
<span style="color:black; font-size: 12pt"></span>

Alexey Romanenko,
<font color="blue">alexromsput@gmail.com</font>

**Key words:**
    - Loss functions: MAPE, MAPPE, WAPE, WAPPE, WAMAXPE, Hold-out period
    
**Your feedback:**  please provide you feedback  <a href="https://forms.gle/bEV5iwmTqdM9rUoy8"> here </a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import timedelta
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa import stattools
from copy import deepcopy

# don't forget to upload utils file to the colab session
# from utils import qualityMACAPE, qualityRMSE
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

# setting of the plotting style, registers pandas date converters for matplotlib and the default figure size
import seaborn as sns
sns.set_style("darkgrid")
pd.plotting.register_matplotlib_converters()
# Default figure size
sns.mpl.rc("figure", figsize=(25, 5))
sns.mpl.rc("font", size=14)

## Loss Functions for Time Series Forecasting
**Questions**
  * What loss functions for regression tasks do you know?

$$  SSE =\sum_{t=1}^T  \left({A_t-F_t}\right)^2 $$
$$  MSE =\frac{1}{T}\sum_{t=1}^T  \left({A_t-F_t}\right)^2 $$
$$  RMSE =\sqrt{\frac{1}{T}\sum_{t=1}^T  \left({A_t-F_t}\right)^2} $$
$$  MAE =\frac{1}{T}\sum_{t=1}^T  \left|{A_t-F_t}\right| $$
$$  MAPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right| $$
$$  MAPPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{F_t}\right| $$
$$  SMAPE =\frac{2}{T}\sum_{t=1}^T  \frac{|A_t-F_t|}{|A_t|+|F_t|} $$
$$  MAMAXPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{\max(|A_t|,|F_t|)}\right| $$
$$  MASE = \frac{\frac{1}{T}\sum_{t=1}^T |A_t-F_t|}{\frac{1}{N-m}\sum_{i=m+1}^N |z_i-z_{i-m}|} $$
<br></br>
$$  MedianAE = Median \bigl(\left|A_1-F_1\right|,\dots,\left|A_T-F_T\right|\bigr) $$
<br></br>
____________________________________________________________________________
**Closer to Business Metrics**
* from MAPE to WAPE:
   $$  MAPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right| =\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right|*\underbrace{\frac{1}{T}}_{\color{red}w} $$
where $w$ is a weight of observation in the moment $t$
* Let's assume that $w_t = \frac{|A_t|}{\sum_{t=1}^T |A_t|}$

$$  WAPE =\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right|*w=\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right|*\frac{|A_t|}{\sum_{t=1}^T |A_t|} = \sum_{t=1}^T  \frac{\left|A_t-F_t \right|}{\color{red}{|A_t|}}*\frac{\color{red}{|A_t|}}{\sum_{t=1}^T |A_t|} = \frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\left|A_t\right|}  $$

<br></br>
$$  WAPE = \frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\left|A_t\right|} $$


$$  WAPPE =\frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\left|F_t\right|} $$
$$  WAMAXPE =\frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\max(\left|A_t\right|,\left|F_t\right| )} $$

* Note: usually, $sum()$ index includes time dimension, as well as organizational dimensions (product, locations, customers, clients, channels, etc.).  

In [ ]:
# One tested implementation for the lesson and the rest of the course.
# Upload utils.py alongside this notebook when using Colab.
from utils import (
    qualitySSE, qualityMSE, qualityRMSE, qualityMAE, qualityMAPE,
    qualityMAPPE, qualitySMAPE, qualityMAMAXPE, qualityMASE,
    qualityMedianAE, qualityWAPE, qualityWAPPE, qualityWAMAXPE,
)


### Metric conventions and a hand-checkable example

MASE uses a **separate training series** $z_1,\ldots,z_N$. Set $m=1$ for
non-seasonal data or the seasonal period (for example, $m=7$ for weekly
seasonality in daily data). Never estimate its scale on the evaluation period.

All ratios here are fractions (0.1 means 10%). DataFrames are scored per column.
Only pairs with finite actuals and forecasts are evaluated; indexes and columns
must match. Missing training observations do not close gaps in the MASE calendar.

- MAPE at a zero actual and MAPPE at a zero forecast are undefined: these entries
  are NaN and excluded from the mean. Always report the number of evaluated pairs.
- SMAPE uses $|A_t|+|F_t|$; MAMAXPE uses the elementwise maximum of absolute values.
  For these two metrics we define the contribution of an exact zero/zero pair as 0.
- WAPE/WAPPE/WAMAXPE with a zero total denominator and MASE with a zero training
  scale return NaN, not zero. MASE is therefore undefined for constant training
  data (or seasonally constant data at the chosen period).

Reference: [Forecasting: Principles and Practice — accuracy](https://otexts.com/fpp3/accuracy.html).


In [ ]:
# Training naive errors: 2, 2, 4 -> scale = 8/3.
# Evaluation absolute errors: 2, 4 -> MAE = 3 -> MASE = 9/8.
train_example = pd.Series([2., 4., 6., 10.])
actual_example = pd.Series([20., 30.])
forecast_example = pd.Series([18., 26.])
mase, scaled_errors = qualityMASE(actual_example, forecast_example, train=train_example)
assert np.isclose(mase, 1.125)
assert np.allclose(scaled_errors, [0.75, 1.5])
assert np.isnan(qualityMASE(actual_example, forecast_example, train=pd.Series([5., 5., 5.]))[0])
assert np.isnan(qualityWAPE(pd.Series([0., 0.]), pd.Series([0., 0.]))[0])
assert qualitySMAPE(pd.Series([0., -2.]), pd.Series([0., 2.]))[0] == 1.0
print('MASE =', mase, '; all hand-checkable examples passed')


In [ ]:
length = 500
frc_ts = pd.DataFrame([[1+x, length-(1+x)**(0.9)-100] for x in range(length)], columns = ['x','y'])
frc_ts.plot().update_layout(height=350, width=1300)

In [ ]:
# fig = plt.figure(figsize = (25,35))
# ax1=fig.add_subplot(111)
qlt_array=['SSE', 'MSE', 'RMSE', 'MAE', 'MAPE', 'MAPPE', 'SMAPE', 'MASE', 'MedianAE', 'WAPE', 'WAPPE', 'WAMAXPE']
qlt = [np.nan]*len(qlt_array)
fig = make_subplots(rows=4, cols=3)

qlt_desc = [r"$SSE =\sum_{t=1}^T  \left({A_t-F_t}\right)^2 $",
r"$MSE =\frac{1}{T}\sum_{t=1}^T  \left({A_t-F_t}\right)^2 $",
r"$RMSE =\sqrt{\frac{1}{T}\sum_{t=1}^T  \left({A_t-F_t}\right)^2} $",
r"$MAE =\frac{1}{T}\sum_{t=1}^T  \left|{A_t-F_t}\right| $",
r"$MAPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{A_t}\right| $",
r"$MAPPE =\frac{1}{T}\sum_{t=1}^T  \left|\frac{A_t-F_t}{F_t}\right| $",
r"$SMAPE =\frac{2}{T}\sum_{t=1}^T  \frac{|A_t-F_t|}{|A_t|+|F_t|} $",
r"$MASE = \frac{\frac{1}{T}\sum_{t=1}^T |A_t-F_t|}{\frac{1}{N-m}\sum_{i=m+1}^N |z_i-z_{i-m}|} $",
r"$MedianAE = Median \left(\left|A_1-F_1\right|,\dots,\left|A_T-F_T\right|\right) $",
r"$WAPE =\frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\left|A_t\right|} $",
r"$WAPPE =\frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\left|F_t\right|} $",
r"$WAMAXPE =\frac{\sum_{t=1}^T\left|A_t-F_t\right|}{\sum_{t=1}^T\max(\left|A_t\right|,\left|F_t\right| )} $"]


# Independent training history for the MASE denominator (scale = 1).
mase_train = pd.Series(np.arange(-length, 0, dtype=float))

for i in range(len(qlt_array)):
  kwargs = {'train': mase_train} if qlt_array[i] == 'MASE' else {}
  qlt[i], qlt_vec = globals()['quality'+qlt_array[i]](frc_ts['x'], frc_ts['y'], **kwargs)
  qlt_vec.index = frc_ts['x']-frc_ts['y']
  fig.add_trace(
      go.Scatter(x=(frc_ts['x']-frc_ts['y']), y=qlt_vec,
      name=qlt_desc[i]),
      row=i//3+1, col=i%3+1
  )
fig.show()

**Let's consider random signal $y_t: {\bf{E}}(y_t, y_{t+\tau}) = 0, \forall \tau>0$**.

Optimal forecast:
$$\hat y^* = \arg\min_{\hat y}\left(\int_{-\infty}^{+\infty}Loss(y, \hat y)\cdot~p(y) dy\right)$$

**Questions**:
  * What is the optimal forecast for prediction of such $y_t$?
  * What is the optimal forecast for prediction of such $y_t$ w.r.t. SSE (RMSE) loss function?
  
  $\hat y^* = {\mathbf E}y$

  * What is the optimal forecast for prediction of such $y_t$ w.r.t. MAE loss function?

  $\hat y^* = F^{-1}(0.5)~-$ median

  * What is the optimal forecast for prediction of such $y_t$ w.r.t. MAPE/MAPPE/WAPE/WAPPE/WAMAXPE loss functions?
  

In [ ]:
# generate random variable
shape, scale = 2., 2.  # mean=2, std=2*sqrt(2)
y = np.random.gamma(shape, scale, 1000)
pd.DataFrame(y).plot().update_layout(height=350, width=1300)

In [ ]:
# self-practice task: realize search for optimal value wrt to all loss functions above
from scipy.optimize import minimize
# def rosen(x):
#     """The Rosenbrock function"""
#     return sum(100.0*(x[1:]-x[:-1]**2.0)**2.0 + (1-x[:-1])**2.0)

def quality_real_set(x=0, qlt='WAPE'):
  return  eval('quality'+qlt)(pd.DataFrame(y, columns=['Y'])['Y'], pd.DataFrame([x]*len(y), columns=['X'])['X'])[0]

x0 = 0.1
res = minimize(quality_real_set, x0, 'MAE')
print(res.x)

In [ ]:
# plot histogramm
fig = px.histogram(pd.DataFrame(y, columns = ['ts values']), x='ts values')
for qlt_func in ['WAPPE', 'WAPE', 'RMSE', 'MAE', 'WAMAXPE']:
  fig.add_vline(x=minimize(quality_real_set, x0, qlt_func).x[0], line_dash = 'dash', line_color = 'firebrick', annotation_text=qlt_func)
# fig.add_vline(x=np.mean(y), line_dash = 'dash', line_color = 'blue', annotation_text=qlt_func)

fig.update_layout(height=350, width=1300).show()

<img src = 'https://github.com/aromanenko/ATSF/blob/main/pic/Quality%20Comparison.png?raw=true'>

## Selecting the Best Time Series Forecasting Algorithm


**Two-steps CV approach**
<img src = "https://www.researchgate.net/publication/307087929/figure/fig5/AS:399685689856005@1472303902481/In-the-Cross-validation-and-testing-approach-the-data-are-divided-into-two-separate.png">

<font size=5pt> Question </font>

**How much can cross-validation score be different from real-correct score?**
  
* See example <a href = 'https://github.com/aromanenko/ATSF/blob/main/two-steps%20cross-validation.ipynb'>here </a>

In [ ]:
test_period_start = '2016-01-01'
test_period_end = '2018-01-01'
loss = 'SMAPE'

In [ ]:
# Wage
ts_wage = pd.read_csv('https://raw.githubusercontent.com/aromanenko/ATSF/33b78c9be63831567f8036ad40c5859e824d30f7/data/monthly-wage.csv', sep=';', decimal='.', parse_dates=True, index_col='Month')
ts_wage.index.names=['Timestamp']
ts_wage = ts_wage['01-01-1999':]
ts_wage.plot()

In [ ]:
# forecasting with prophet
# !pip install prophet

from prophet import Prophet
#suppressing the info logs
import logging
logging.getLogger('fbprophet').setLevel(logging.ERROR)

#creating a dataframe that fbprophet requires
# df = pd.DataFrame([col])
# df['ds'] = df.index
# df.columns = ['y', 'ds']

#making predictions
m = Prophet()
m.fit(ts_wage[:test_period_start].reset_index().rename(columns={'Real wage':'y', 'Timestamp':'ds'}))
#'make_future_dataframe' parameter decides how far into the future we're looking
future = m.make_future_dataframe(12,freq = 'M')
forecast = m.predict(future)

test_loss = eval('quality' + loss)(ts_wage.loc[test_period_start:test_period_end, 'Real wage'], forecast.set_index('ds').loc[test_period_start:test_period_end, 'yhat'])[0]

#plotting the results
plt.figure()
plt.plot(ts_wage.loc[:test_period_start].index, ts_wage.loc[:test_period_start], label='Train')
plt.plot(ts_wage.loc[test_period_start:].index, ts_wage.loc[test_period_start:, 'Real wage'], label='Test')
plt.plot(forecast.set_index('ds').loc[test_period_start:].index, forecast.set_index('ds').loc[test_period_start:,'yhat'], color='green', alpha=0.7, label='fbprophet')
plt.title('wage forecast by fbprphet, test period {0} = {1}'.format(loss, eval('quality' + loss)(ts_wage.loc[test_period_start:test_period_end, 'Real wage'], forecast.set_index('ds').loc[test_period_start:test_period_end, 'yhat'])[0]))
plt.legend()

In [ ]:
# statmodels TW
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train, test = ts_wage.loc[:test_period_start, 'Real wage'], ts_wage.loc[test_period_start:, 'Real wage']
model = ExponentialSmoothing(train, trend = 'add', seasonal='add', seasonal_periods=12).fit()
twpred = model.predict(start=test.index[0], end=test.index[-1])

plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test')
plt.plot(twpred.index, twpred, label='Theil-Wage')
plt.title('wage forecast by smThailWage, test period {0} = {1}'.format(loss, eval('quality' + loss)(test, twpred)[0]))
plt.legend(loc='best')


In [ ]:
# statmodels SARIMAX
def fit_arima_12(ts, p,d,q,P,D,Q, exogs_var = None):
    try:
        mod = sm.tsa.statespace.SARIMAX(endog = ts, order=(p,d,q), seasonal_order=(P,D,Q,12), exog = exogs_var)
        results = mod.fit()
        return("sarimax_%i%i%i_%i%i%i: aic=%f, bic=%f, hqic=%f " % (p,d,q,P,D,Q,
               results.aic, results.bic, results.hqic))
    except Exception:
        return ("sarimax_%i%i%i_%i%i%i: can't fit model " % (p,d,q,P,D,Q))


# Qaution: we assume that the appropriate d = 1, D = 1
# try most popular parameters of ARIMA
from sklearn.model_selection import ParameterGrid

param_grid = {'p': [0, 1], 'd' : [1],
             'q': [0, 1], 'P' : [0, 1],
             'D': [1], 'Q' : [0,1]}

grid = ParameterGrid(param_grid)

for params in grid:
    print(fit_arima_12(ts_wage.loc[:test_period_start, 'Real wage'],params['p'], params['d'], params['q'], params['P'], params['D'], params['Q']))

In [ ]:
sarimax_110_010 = sm.tsa.statespace.SARIMAX(endog= ts_wage.loc[:test_period_start, 'Real wage'], trend='n',
                                            order=(1,1,0), seasonal_order=(0,1,0,12)).fit()
arima_frc =sarimax_110_010.forecast(steps = 12)

#plotting the results
plt.figure()
plt.plot(ts_wage.loc[:test_period_start].index, ts_wage.loc[:test_period_start], label='Train')
plt.plot(ts_wage.loc[test_period_start:].index, ts_wage.loc[test_period_start:, 'Real wage'], label='Test')
plt.plot(arima_frc.loc[test_period_start:].index, arima_frc.loc[test_period_start:], color='green', alpha=0.7, label='SARIMAX')
plt.title('wage forecast by sm sarimax, test period {0} = {1}'.format(loss, eval('quality' + loss)(ts_wage.loc[test_period_start:test_period_end, 'Real wage'], arima_frc.loc[test_period_start:test_period_end])[0]))
plt.legend()

## (self study) How to define target forecast accuracy?
**Question**
 - What drives big loss of ts forecast?

**Answer**
 - noise level
 - causal variables (dependance on exogenous variables)
 - seasonality patterns
 - trend
 - intermittency
 - environment


### Building Time Series Forecastability Index
Those Time Series that have about the same accuracy level should have the same $\color{red}{Forecastability~Index}$.



<img src = 'https://github.com/aromanenko/ATSF/blob/main/pic/Forecastability%20Index.png?raw=true' width = 500pt>

Techniques to do it
* STL decomposition (detecting trend and seasonality) https://www.statsmodels.org/devel/examples/notebooks/generated/stl_decomposition.html

* ABC splitting (to segment by Demand Velocity and Demand Variability) https://pypi.org/project/abc-analysis/

or <a href = 'https://excel.city/2015/01/methods-of-classification-in-abc-analysis/'>more advanced tecniques </a>

* Intermittency time series tests


# Conclusion
## Self-check questions
  1. Which loss-functions stimulate UnderForecasting issues?
  2. Which loss-functions stimulate Overfrecasting issues?
  3. What is the difference between intermittent time series and highly promo drive one?
  4. What is the idea of Croston method?
  5. Describe all steps in Training Time Series Forecasting Algorithims Approach


**Materials**
* Hyndman R.J., Athanasopoulos G. Forecasting: principles and practice,2016. https://www.otexts.org/book/fpp
* https://us.nttdata.com/en/blog/2021/december/understanding-forecastability-and-how-to-generate-better-forecasts
* Two-steps cross-validation https://scikit-learn.org/1.5/modules/cross_validation.html